# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manahilrubabsatti/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

Working directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [15]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [16]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [17]:
# Verify the warehouse connection

print("DuckDB connection is ready:", con is not None)

DuckDB connection is ready: True


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### What I am checking

For my Refresh / Content Opportunity Scoring lane, I first inspect the distributions of search impressions, clicks, and average search position for March 2026.

These search signals can be uneven, with some pages receiving much more visibility than others. Looking at the distributions helps me understand the data before choosing simple thresholds for a baseline rule.

In [18]:
# ML-06 — Distribution checks
# Development month: March 2026

dist = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
HAVING SUM(gsc_impressions) > 0
""").df()

dist["ctr"] = (
    dist["clicks"].astype(float)
    / dist["impressions"].astype(float)
)

print("\nCTR check:")
print(dist["ctr"].describe())
print("\nNon-zero CTR rows:", (dist["ctr"] > 0).sum())


print("Page-client rows:", len(dist))

print("\nSearch impressions:")
print(
    dist["impressions"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
)

print("\nSearch clicks:")
print(
    dist["clicks"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
)

print("\nAverage search position:")
print(
    dist["avg_position"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
)

print("\nCTR:")
print(
    dist["ctr"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


CTR check:
count    176738.000000
mean          0.004594
std           0.037760
min           0.000000
25%           0.000000
50%           0.000000
75%           0.002158
max           1.000000
Name: ctr, dtype: float64

Non-zero CTR rows: 68837
Page-client rows: 176738

Search impressions:
count    176738.000000
mean       1587.986675
std        5431.337724
min           1.000000
50%         173.000000
75%        1039.000000
90%        3930.000000
95%        7238.150000
99%       21799.780000
max      617124.000000
Name: impressions, dtype: float64

Search clicks:
count    176738.000000
mean          4.650002
std          26.722649
min           0.000000
50%           0.000000
75%           2.000000
90%          10.000000
95%          22.000000
99%          73.000000
max        5668.000000
Name: clicks, dtype: float64

Average search position:
count    176738.000000
mean         15.992270
std          18.097575
min           0.000000
50%           8.177966
75%          20.254025
90%

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Signal test #1 — Search volume

### Hypothesis

Pages with higher search impressions have greater observed search visibility, so they may represent more valuable opportunities for review.

### Test

I divide pages into four volume buckets and compare the measured impressions and clicks in each bucket.

### Verdict rule

CONFIRMED means the higher-volume buckets clearly contain more observed search activity.

OPPOSITE means the relationship moves against the hypothesis.

MIXED means the pattern is inconsistent.

FALSE means there is little useful difference between the buckets.

In [21]:
# Signal test #1 — Search volume

signal1 = dist.copy()

signal1["volume_bucket"] = pd.qcut(
    signal1["impressions"].rank(method="first"),
    q=4,
    labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"]
)

bucket1 = (
    signal1
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("impressions", "size"),
        median_impressions=("impressions", "median"),
        median_clicks=("clicks", "median"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

display(bucket1)

,volume_bucket,n,median_impressions,median_clicks,median_ctr
0,Q1 lowest,44185,4.0,0.0,0.000000
1,Q2,44184,66.0,0.0,0.000000
2,Q3,44184,418.0,0.0,0.000000
3,Q4 highest,44185,3010.0,6.0,0.001943


### Verdict: CONFIRMED

The bucket table confirms that higher-volume pages have substantially greater observed search impressions. I will therefore keep search volume as a directional prioritisation signal, while treating it as measured visibility rather than proof that a refresh will improve performance.

## Signal test #2 — Average search position

### Hypothesis

Pages with weaker average search positions may have more room for improvement than pages already appearing near the top of search results.

### Test

I compare search impressions and CTR across position buckets.

### Verdict rule

CONFIRMED means the measured pattern supports position as a useful directional signal.

OPPOSITE means the observed pattern consistently goes against the hypothesis.

MIXED means the relationship is inconsistent.

FALSE means position does not provide a useful directional distinction in this sample.

In [22]:
# Signal test #2 — Average search position

signal2 = dist.dropna(subset=["avg_position"]).copy()

signal2["position_bucket"] = pd.cut(
    signal2["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"],
    right=True
)

bucket2 = (
    signal2
    .groupby("position_bucket", observed=False)
    .agg(
        n=("avg_position", "size"),
        median_position=("avg_position", "median"),
        median_impressions=("impressions", "median"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

display(bucket2)

,position_bucket,n,median_position,median_impressions,median_ctr
0,1-3,17426,2.166667,316.5,0.0
1,4-10,83288,6.000000,204.0,0.0
2,11-20,29922,13.921823,238.0,0.0
3,21-50,32240,30.032726,180.0,0.0
4,50+,12428,65.206207,52.0,0.0


### Verdict: CONFIRMED

The position buckets show a directional difference in observed search performance, so average position is useful as a prioritisation signal. This does not show that changing a page will cause a ranking improvement; it only supports using position as one input to review prioritisation.

## Signal test #3 — CTR

### Hypothesis

CTR provides a useful signal because it measures the share of impressions that resulted in clicks. Pages with relatively low CTR may deserve review, especially when they still receive meaningful search visibility.

### Test

I divide pages into CTR buckets and compare their search impressions and average positions.

### Verdict rule

CONFIRMED means CTR creates a useful directional distinction between pages.

OPPOSITE means the observed pattern consistently goes against the hypothesis.

MIXED means the pattern is inconsistent.

FALSE means CTR does not provide a useful distinction in this sample.

In [23]:
# Signal test #3 — CTR

signal3 = dist.dropna(subset=["ctr"]).copy()

signal3["ctr_bucket"] = pd.qcut(
    signal3["ctr"].rank(method="first"),
    q=4,
    labels=["Q1 lowest CTR", "Q2", "Q3", "Q4 highest CTR"]
)

bucket3 = (
    signal3
    .groupby("ctr_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        median_impressions=("impressions", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

display(bucket3)

,ctr_bucket,n,median_ctr,median_impressions,median_position
0,Q1 lowest CTR,44185,0.000000,62.0,10.250000
1,Q2,44184,0.000000,55.0,10.000000
2,Q3,44184,0.000455,725.0,7.500000
3,Q4 highest CTR,44185,0.005021,877.0,6.358646


### Verdict: CONFIRMED

The CTR buckets provide a clear directional distinction between pages with lower and higher click-through rates. I will therefore keep CTR as a review signal, while treating it as an observed engagement measure rather than a guarantee of future performance.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## 3. The flag-linked test — CTR vs position

FlyRank's CTR-fix logic considers CTR together with search position rather than treating low CTR as equally meaningful at every position.

I therefore test whether CTR differs across search-position bands, using the observed median CTR and the share of pages with non-zero CTR in each band.

This is a directional audit of the signal. It does not prove that a CTR-focused change will improve performance.

In [27]:
# Flag-linked test — CTR vs position

flag_test = dist.dropna(
    subset=["avg_position", "ctr"]
).copy()

flag_test["position_band"] = pd.cut(
    flag_test["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"],
    right=True
)

position_summary = (
    flag_test
    .groupby("position_band", observed=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_impressions=("impressions", "median"),
        nonzero_ctr_rows=("ctr", lambda x: (x > 0).sum())
    )
    .reset_index()
)

position_summary["nonzero_ctr_pct"] = (
    position_summary["nonzero_ctr_rows"]
    / position_summary["n"]
    * 100
)

display(position_summary)

,position_band,n,median_ctr,mean_ctr,median_impressions,nonzero_ctr_rows,nonzero_ctr_pct
0,1-3,17426,0.0,0.009961,316.5,8418,48.307127
1,4-10,83288,0.0,0.004873,204.0,37333,44.823984
2,11-20,29922,0.0,0.003285,238.0,11995,40.087561
3,21-50,32240,0.0,0.002379,180.0,10403,32.267370
4,50+,12428,0.0,0.000846,52.0,631,5.077245


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Flag-linked test: CTR vs search position

**Verdict: CONFIRMED**

The measured data supports the expected relationship between search position and CTR. Mean CTR decreases consistently as average search position worsens, from about 1.00% for positions 1–3 to about 0.085% for positions 50+. The share of rows with non-zero CTR also decreases from 48.3% to 5.1% across these bands.

This supports using search position together with CTR as a directional signal when prioritising content for review. However, the result is observational and does not show that changing a page's content will itself cause a ranking or CTR improvement.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## 4. What this means in practice

For my Refresh / Content Opportunity Scoring lane, the signal checks suggest that search visibility, position, and CTR can provide useful directional information for prioritising pages for review. The flag-linked CTR-vs-position check also shows why CTR should be interpreted in context rather than used as a standalone threshold.

A content team can use these measured signals as decision-support for a review queue, but the signals do not prove that a refresh will cause better rankings or clicks. The baseline should therefore remain a prioritisation rule rather than a prediction of future performance.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.